# Core Concepts: Domains, Units, and Coordinates

This tutorial introduces the fundamental building blocks of TimeToAlign!: Domains, TimeUnits, Coordinates, and Timelines.

**Learning Objectives:**
- Understand the three temporal domains (Physical, Logical, Graphical)
- Work with TimeUnits and their domain compatibility
- Create and manipulate Coordinate objects
- Know the six Timeline types and when to use each

**Prerequisites:**
- Basic Python knowledge
- TimeToAlign! installed (`pip install timetoalign`)

## Why These Concepts Matter

Music exists in multiple representations simultaneously:

- **A score** represents music in symbolic notation (beats, measures, quarter notes)
- **An audio recording** represents music in physical time (seconds, samples)
- **A score image** represents music in visual space (pixels, coordinates)

TimeToAlign! provides a unified framework for working across these representations. The key insight is that all musical time can be described using a small set of concepts:

1. **Domain**: Which representation are we working with?
2. **TimeUnit**: What are we measuring in?
3. **Coordinate**: A specific position within a domain
4. **Timeline**: An ordered axis of coordinates

## Setup

In [1]:
# Standard imports
from fractions import Fraction

# TimeToAlign! core imports
import timetoalign as tta
from timetoalign import Domain, TimeUnit, NumberType, Coordinate

print(f"TimeToAlign! version: {tta.__version__}")

TimeToAlign! version: 0.1.0


---

## The Three Domains

TimeToAlign! organizes all temporal data into three **Domains**:

| Domain | Description | Examples |
|--------|-------------|----------|
| **Physical** | Real-world time, audio | Seconds, milliseconds, samples |
| **Logical** | Symbolic, musical | Beats, quarters, ticks, measures |
| **Graphical** | Visual, spatial | Pixels, centimeters, inches |

Each domain represents a fundamentally different way of conceptualizing "time" in music.

In [2]:
# List all domains
print("Available Domains:")
print("="*40)
for domain in Domain:
    print(f"  Domain.{domain.name:<10} = {domain.value!r}")

Available Domains:
  Domain.logical    = 'logical'
  Domain.physical   = 'physical'
  Domain.graphical  = 'graphical'


In [3]:
# Domains have convenient aliases for quick access
print("Domain Aliases:")
print(f"  Domain.physical  == Domain.ph  -> {Domain.physical == Domain.ph}")
print(f"  Domain.logical   == Domain.lo  -> {Domain.logical == Domain.lo}")
print(f"  Domain.graphical == Domain.gr  -> {Domain.graphical == Domain.gr}")

Domain Aliases:
  Domain.physical  == Domain.ph  -> True
  Domain.logical   == Domain.lo  -> True
  Domain.graphical == Domain.gr  -> True


In [4]:
# Domains can be constructed from strings
print("Creating Domains from strings:")
print(f"  Domain('physical')  -> {Domain('physical')}")
print(f"  Domain('lo')        -> {Domain('lo')}")
print(f"  Domain('graphical') -> {Domain('graphical')}")

Creating Domains from strings:
  Domain('physical')  -> physical
  Domain('lo')        -> logical
  Domain('graphical') -> graphical


### Understanding Each Domain

**Physical Domain** (`Domain.physical`)
- Represents real-world, wall-clock time
- Used for audio files, recordings, performances
- Units: seconds, milliseconds, samples, frames

**Logical Domain** (`Domain.logical`)
- Represents symbolic, musical time
- Used for scores, MIDI files, notation
- Units: beats, quarters, measures, ticks

**Graphical Domain** (`Domain.graphical`)
- Represents visual, spatial coordinates
- Used for score images, sheet music PDFs, spectrograms
- Units: pixels, centimeters, inches, points

---

## TimeUnits

A **TimeUnit** specifies the measuring unit for coordinates. Each unit belongs to exactly one domain.

In [5]:
# List all TimeUnits grouped by domain
print("TimeUnits by Domain:")
print("="*60)

for domain in Domain:
    print(f"\n{domain.name.upper()}:")
    units = [u for u in TimeUnit if u.domain == domain]
    for unit in units:
        discrete = "(discrete)" if unit.is_discrete else ""
        print(f"  TimeUnit.{unit.name:<14} {discrete}")

TimeUnits by Domain:

LOGICAL:
  TimeUnit.number         
  TimeUnit.beats          
  TimeUnit.measures       
  TimeUnit.quarters       
  TimeUnit.ticks          (discrete)

PHYSICAL:
  TimeUnit.milliseconds   
  TimeUnit.seconds        
  TimeUnit.minutes        
  TimeUnit.samples        (discrete)
  TimeUnit.frames         (discrete)

GRAPHICAL:
  TimeUnit.pixels         (discrete)
  TimeUnit.meters         
  TimeUnit.centimeters    
  TimeUnit.millimeters    
  TimeUnit.inches         
  TimeUnit.points         


In [6]:
# Each TimeUnit knows its domain and discreteness
examples = [TimeUnit.seconds, TimeUnit.quarters, TimeUnit.ticks, TimeUnit.pixels]

print("TimeUnit Properties:")
print("="*50)
print(f"{'Unit':<16} {'Domain':<12} {'Discrete?'}")
print("-"*50)
for unit in examples:
    print(f"{unit.name:<16} {unit.domain.name:<12} {unit.is_discrete}")

TimeUnit Properties:
Unit             Domain       Discrete?
--------------------------------------------------
seconds          physical     False
quarters         logical      False
ticks            logical      True
pixels           graphical    True


### Discrete vs. Continuous Units

Units can be **discrete** (countable, integer values) or **continuous** (any real number):

| Continuous | Discrete |
|------------|----------|
| seconds (1.5s) | samples (44100) |
| quarters (2.5q) | ticks (480) |
| centimeters (3.2cm) | pixels (1920) |

This distinction matters for arithmetic and timeline types.

In [7]:
# TimeUnits have convenient aliases
print("TimeUnit Aliases:")
print(f"  TimeUnit.seconds      == TimeUnit.s   -> {TimeUnit.seconds == TimeUnit.s}")
print(f"  TimeUnit.milliseconds == TimeUnit.ms  -> {TimeUnit.milliseconds == TimeUnit.ms}")
print(f"  TimeUnit.quarters     == TimeUnit.q   -> {TimeUnit.quarters == TimeUnit.q}")
print(f"  TimeUnit.beats        == TimeUnit.b   -> {TimeUnit.beats == TimeUnit.b}")
print(f"  TimeUnit.pixels       == TimeUnit.px  -> {TimeUnit.pixels == TimeUnit.px}")
print(f"  TimeUnit.ticks        == TimeUnit.pulses  -> {TimeUnit.ticks == TimeUnit.pulses}")

TimeUnit Aliases:
  TimeUnit.seconds      == TimeUnit.s   -> True
  TimeUnit.milliseconds == TimeUnit.ms  -> True
  TimeUnit.quarters     == TimeUnit.q   -> True
  TimeUnit.beats        == TimeUnit.b   -> True
  TimeUnit.pixels       == TimeUnit.px  -> True
  TimeUnit.ticks        == TimeUnit.pulses  -> True


---

## NumberType

TimeToAlign! supports three numeric types for coordinate values:

| Type | Python Type | Use Case |
|------|-------------|----------|
| `NumberType.int` | `int` | Discrete units (samples, ticks, pixels) |
| `NumberType.float` | `float` | Physical time (seconds, milliseconds) |
| `NumberType.fraction` | `Fraction` | Exact rational values (beats, quarters) |

The `Fraction` type is particularly important for musical time, where triplets, dotted notes, and complex subdivisions require exact representation.

In [8]:
# List all NumberTypes
print("NumberTypes:")
print("="*40)
for nt in NumberType:
    print(f"  NumberType.{nt.name:<10} -> {nt.python_type.__name__}")

NumberTypes:
  NumberType.int        -> int
  NumberType.float      -> float
  NumberType.fraction   -> Fraction


In [9]:
# NumberType can be inferred from values
print("NumberType Inference:")
print(f"  NumberType.from_number(42)           -> {NumberType.from_number(42)}")
print(f"  NumberType.from_number(3.14)         -> {NumberType.from_number(3.14)}")
print(f"  NumberType.from_number(Fraction(3,4)) -> {NumberType.from_number(Fraction(3,4))}")

NumberType Inference:
  NumberType.from_number(42)           -> int
  NumberType.from_number(3.14)         -> float
  NumberType.from_number(Fraction(3,4)) -> fraction


### Why Fractions Matter for Musical Time

Consider a triplet quarter note. Its duration is exactly 2/3 of a beat. Using floats:

```python
2/3 = 0.6666666666666666  # Imprecise!
```

Using fractions:

```python
Fraction(2, 3) = 2/3  # Exact!
```

This precision is essential when aligning events across timelines.

In [10]:
# Demonstrating precision differences
triplet_float = 2/3
triplet_fraction = Fraction(2, 3)

print("Triplet Quarter Note Duration:")
print(f"  Float:    {triplet_float}")
print(f"  Fraction: {triplet_fraction}")

# Adding three triplets should equal 2 beats
print("\nThree triplets (should equal 2):")
print(f"  Float sum:    {triplet_float * 3}")
print(f"  Fraction sum: {triplet_fraction * 3}")

Triplet Quarter Note Duration:
  Float:    0.6666666666666666
  Fraction: 2/3

Three triplets (should equal 2):
  Float sum:    2.0
  Fraction sum: 2


---

## Coordinates

A **Coordinate** is the fundamental building block of TimeToAlign!. It pairs a numeric value with a TimeUnit.

```python
Coordinate(value, unit)
```

Coordinates are:
- **Immutable** (frozen dataclass)
- **Hashable** (can be used in sets and as dict keys)
- **Type-safe** (arithmetic respects units)

In [11]:
# Creating coordinates
c1 = Coordinate(120, TimeUnit.ticks)
c2 = Coordinate(1.5, TimeUnit.seconds)
c3 = Coordinate(Fraction(3, 4), TimeUnit.quarters)

print("Created Coordinates:")
print(f"  c1 = {c1}")
print(f"  c2 = {c2}")
print(f"  c3 = {c3}")

Created Coordinates:
  c1 = 120 ticks
  c2 = 1.5 seconds
  c3 = 3/4 quarters


In [12]:
# Coordinate properties
print(f"Coordinate: {c3}")
print(f"  .value       = {c3.value}")
print(f"  .unit        = {c3.unit}")
print(f"  .number_type = {c3.number_type}")
print(f"  .domain      = {c3.domain}")

Coordinate: 3/4 quarters
  .value       = 3/4
  .unit        = quarters
  .number_type = fraction
  .domain      = logical


In [13]:
# String representations
print("String Representations:")
print(f"  str(c1):  {str(c1)}")
print(f"  repr(c1): {repr(c1)}")
print()
print(f"  str(c3):  {str(c3)}")
print(f"  repr(c3): {repr(c3)}")

String Representations:
  str(c1):  120 ticks
  repr(c1): Coordinate(120, ticks)

  str(c3):  3/4 quarters
  repr(c3): Coordinate(Fraction(3, 4), quarters)


### Coordinate Arithmetic

Coordinates support arithmetic operations with type safety:

- **Addition/Subtraction**: Between coordinates with the **same unit**
- **Multiplication/Division**: With **scalars** (int, float, Fraction)

In [14]:
# Addition and subtraction (same unit required)
a = Coordinate(100, TimeUnit.ticks)
b = Coordinate(50, TimeUnit.ticks)

print("Addition and Subtraction:")
print(f"  {a} + {b} = {a + b}")
print(f"  {a} - {b} = {a - b}")

Addition and Subtraction:
  100 ticks + 50 ticks = 150 ticks
  100 ticks - 50 ticks = 50 ticks


In [15]:
# Multiplication and division with scalars
c = Coordinate(2, TimeUnit.quarters)

print("Scaling:")
print(f"  {c} * 3   = {c * 3}")
print(f"  {c} / 2   = {c / 2}")
print(f"  {c} // 2  = {c // 2}  (integer division)")
print(f"  {c} * 1.5 = {c * 1.5}")

Scaling:
  2 quarters * 3   = 6 quarters
  2 quarters / 2   = 1.0 quarters
  2 quarters // 2  = 1 quarters  (integer division)
  2 quarters * 1.5 = 3.0 quarters


In [16]:
# Comparison operators
x = Coordinate(10, TimeUnit.seconds)
y = Coordinate(5, TimeUnit.seconds)

print("Comparisons:")
print(f"  {x} > {y}  -> {x > y}")
print(f"  {x} == {y} -> {x == y}")
print(f"  {x} <= {y} -> {x <= y}")

Comparisons:
  10 seconds > 5 seconds  -> True
  10 seconds == 5 seconds -> False
  10 seconds <= 5 seconds -> False


In [17]:
# Unit mismatch raises TypeError
ticks = Coordinate(480, TimeUnit.ticks)
seconds = Coordinate(1.0, TimeUnit.seconds)

try:
    result = ticks + seconds  # This will fail!
except TypeError as e:
    print(f"TypeError: {e}")
    print("\n(This is intentional - you cannot add ticks and seconds directly!)")

TypeError: Cannot add coordinates with different units: ticks vs seconds

(This is intentional - you cannot add ticks and seconds directly!)


### Coordinate Type Conversions

Coordinates can be converted between numeric types:

In [18]:
# Type conversions
c = Coordinate(Fraction(7, 4), TimeUnit.quarters)

print(f"Original: {c}")
print(f"  .to_float()    -> {c.to_float()}")
print(f"  .to_int()      -> {c.to_int()}  (truncates)")
print(f"  .to_fraction() -> {c.to_fraction()}")

Original: 7/4 quarters
  .to_float()    -> 1.75
  .to_int()      -> 1  (truncates)
  .to_fraction() -> 7/4


In [19]:
# Utility methods
zero = Coordinate(0, TimeUnit.beats)
positive = Coordinate(5, TimeUnit.beats)

print("Utility Methods:")
print(f"  {zero}.is_zero()     -> {zero.is_zero()}")
print(f"  {positive}.is_zero() -> {positive.is_zero()}")
print(f"  {positive}.is_positive() -> {positive.is_positive()}")

Utility Methods:
  0 beats.is_zero()     -> True
  5 beats.is_zero() -> False
  5 beats.is_positive() -> True


In [20]:
# Creating modified copies (coordinates are immutable)
original = Coordinate(100, TimeUnit.ticks)

print(f"Original: {original}")
print(f"  .with_value(200) -> {original.with_value(200)}")
print(f"  .with_unit(TimeUnit.samples) -> {original.with_unit(TimeUnit.samples)}")
print()
print(f"Note: .with_unit() does NOT convert - it just changes the label!")

Original: 100 ticks
  .with_value(200) -> 200 ticks
  .with_unit(TimeUnit.samples) -> 100 samples

Note: .with_unit() does NOT convert - it just changes the label!


---

## The Six Timeline Types

Combining the 3 domains with continuous/discrete variants gives us **6 Timeline types**:

| Domain | Continuous | Discrete |
|--------|------------|----------|
| Physical | `ContinuousPhysicalTimeline` | `DiscretePhysicalTimeline` |
| Logical | `ContinuousLogicalTimeline` | `DiscreteLogicalTimeline` |
| Graphical | `ContinuousGraphicalTimeline` | `DiscreteGraphicalTimeline` |

In [21]:
# Import all timeline types
from timetoalign import (
    ContinuousPhysicalTimeline,
    DiscretePhysicalTimeline,
    ContinuousLogicalTimeline,
    DiscreteLogicalTimeline,
    ContinuousGraphicalTimeline,
    DiscreteGraphicalTimeline,
)

print("All 6 Timeline types imported successfully!")

All 6 Timeline types imported successfully!


In [22]:
# Each timeline type has sensible defaults
timeline_classes = [
    ContinuousPhysicalTimeline,
    DiscretePhysicalTimeline,
    ContinuousLogicalTimeline,
    DiscreteLogicalTimeline,
    ContinuousGraphicalTimeline,
    DiscreteGraphicalTimeline,
]

print("Timeline Type Defaults:")
print("="*70)
print(f"{'Class':<32} {'Domain':<12} {'Unit':<12} {'NumberType'}")
print("-"*70)
for cls in timeline_classes:
    # Create a minimal timeline to inspect defaults
    tl = cls(length=1)
    print(f"{cls.__name__:<32} {tl.domain.name:<12} {tl.unit.name:<12} {tl.number_type.name}")

Timeline Type Defaults:
Class                            Domain       Unit         NumberType
----------------------------------------------------------------------
ContinuousPhysicalTimeline       physical     seconds      float
DiscretePhysicalTimeline         physical     samples      int
ContinuousLogicalTimeline        logical      quarters     fraction
DiscreteLogicalTimeline          logical      ticks        int
ContinuousGraphicalTimeline      graphical    centimeters  float
DiscreteGraphicalTimeline        graphical    pixels       int


### Creating Timelines

Timelines require a `length` parameter (or can be created empty):

In [23]:
# Create timelines for different scenarios

# A score timeline (16 quarter notes)
score_tl = ContinuousLogicalTimeline(length=Fraction(16, 1))
print(f"Score Timeline: {score_tl.length}")

# A MIDI timeline (1920 ticks = 4 beats at 480 PPQN)
midi_tl = DiscreteLogicalTimeline(length=1920)
print(f"MIDI Timeline:  {midi_tl.length}")

# An audio timeline (10 seconds)
audio_tl = ContinuousPhysicalTimeline(length=10.0)
print(f"Audio Timeline: {audio_tl.length}")

# A score image timeline (1920 pixels wide)
image_tl = DiscreteGraphicalTimeline(length=1920)
print(f"Image Timeline: {image_tl.length}")

Score Timeline: 16 quarters
MIDI Timeline:  1920 ticks
Audio Timeline: 10.0 seconds
Image Timeline: 1920 pixels


In [24]:
# Timeline properties
print(f"Timeline: {score_tl.id}")
print(f"  .domain      = {score_tl.domain}")
print(f"  .unit        = {score_tl.unit}")
print(f"  .number_type = {score_tl.number_type}")
print(f"  .origin      = {score_tl.origin}")
print(f"  .length      = {score_tl.length}")
print(f"  .is_locked   = {score_tl.is_locked}")
print(f"  .n_events    = {score_tl.n_events}")

Timeline: tl:7
  .domain      = logical
  .unit        = quarters
  .number_type = fraction
  .origin      = 0 quarters
  .length      = 16 quarters
  .is_locked   = False
  .n_events    = 0


### When to Use Each Timeline Type

| Use Case | Timeline Type | Why |
|----------|---------------|-----|
| MusicXML/MEI scores | `ContinuousLogicalTimeline` | Exact beat fractions |
| MIDI files | `DiscreteLogicalTimeline` | Integer tick resolution |
| Audio analysis | `ContinuousPhysicalTimeline` | Floating-point seconds |
| Sample-accurate audio | `DiscretePhysicalTimeline` | Integer samples/frames |
| Score images | `DiscreteGraphicalTimeline` | Integer pixel positions |
| Scalable graphics | `ContinuousGraphicalTimeline` | Real-valued coordinates |

In [25]:
# Timelines can create coordinates in their native unit
print("Creating coordinates from timelines:")
print(f"  score_tl.make_coordinate(4) = {score_tl.make_coordinate(4)}")
print(f"  midi_tl.make_coordinate(480) = {midi_tl.make_coordinate(480)}")
print(f"  audio_tl.make_coordinate(2.5) = {audio_tl.make_coordinate(2.5)}")

Creating coordinates from timelines:
  score_tl.make_coordinate(4) = 4 quarters
  midi_tl.make_coordinate(480) = 480 ticks
  audio_tl.make_coordinate(2.5) = 2.5 seconds


In [26]:
# Get a summary of timeline state
summary = score_tl.summary()

print("Timeline Summary:")
for key, value in summary.items():
    print(f"  {key}: {value}")

Timeline Summary:
  id: tl:7
  class: ContinuousLogicalTimeline
  unit: quarters
  number_type: fraction
  domain: logical
  length: 16
  is_locked: False
  n_events: 0
  n_children: 0
  event_summary: {'count': 0, 'unit': 'quarters', 'number_type': 'fraction', 'temporal_types': {}, 'event_types': {}, 'coordinate_range': None}


---

## Summary

In this tutorial, we covered the foundational concepts of TimeToAlign!:

1. **Domains**: Physical, Logical, and Graphical - three ways of representing musical time
2. **TimeUnits**: Measuring units (seconds, quarters, pixels, etc.) tied to specific domains
3. **NumberType**: Integer, float, or fraction - choose based on required precision
4. **Coordinates**: Immutable value+unit pairs with type-safe arithmetic
5. **Timelines**: Six types combining domain × continuous/discrete

**Key Takeaway:**
> A Coordinate pairs a numeric value with a unit, ensuring type-safe temporal arithmetic. Timelines organize coordinates into a consistent structure for events.

## Next Steps

- **02_loading_data.ipynb**: Load real music data into EventStores
- **03_conversion_maps.ipynb**: Convert coordinates between units and domains

---

## Exercise 1: Coordinate Arithmetic

**Task:** A MIDI file uses 480 ticks per quarter note (PPQN). Calculate:

1. How many ticks is 4 quarter notes?
2. If an event starts at tick 720, what quarter note position is that?
3. What is the duration in ticks of a dotted half note (3 quarter notes)?

**Hints:**
1. Use `Coordinate` objects with `TimeUnit.ticks`
2. Remember: 1 quarter = 480 ticks

<details>
<summary>Solution</summary>

```python
ppqn = 480  # ticks per quarter note

# 1. Four quarter notes in ticks
four_quarters = Coordinate(4 * ppqn, TimeUnit.ticks)
print(f"4 quarters = {four_quarters}")

# 2. Tick 720 in quarters
tick_720 = 720
quarter_position = tick_720 / ppqn
print(f"Tick 720 = {quarter_position} quarters")

# 3. Dotted half note (3 quarters) in ticks
dotted_half = Coordinate(3 * ppqn, TimeUnit.ticks)
print(f"Dotted half = {dotted_half}")
```

</details>

In [27]:
# Your solution here


---

## Exercise 2: Choosing the Right Timeline

**Task:** For each scenario, identify the most appropriate timeline type:

1. Analyzing a WAV audio file at 44.1kHz sample rate with sample-level precision
2. Representing a MuseScore file with triplets and complex rhythms
3. A PNG image of a musical score (1920x1080 pixels)
4. Real-time audio playback position in a media player

<details>
<summary>Solution</summary>

1. **DiscretePhysicalTimeline** - Sample-level precision requires integer sample indices
2. **ContinuousLogicalTimeline** - Triplets need exact fraction representation (Fraction(1,3))
3. **DiscreteGraphicalTimeline** - Pixels are integers
4. **ContinuousPhysicalTimeline** - Playback position is floating-point seconds

```python
# 1. Audio at sample level
audio_samples = DiscretePhysicalTimeline(length=44100*10)  # 10 seconds
print(f"Audio: {audio_samples.unit}, {audio_samples.number_type}")

# 2. Score with triplets
score = ContinuousLogicalTimeline(length=Fraction(32))  # 32 quarters
triplet = score.make_coordinate(Fraction(1, 3))
print(f"Triplet position: {triplet}")

# 3. Score image
image = DiscreteGraphicalTimeline(length=1920)
print(f"Image: {image.unit}")

# 4. Playback
playback = ContinuousPhysicalTimeline(length=180.0)  # 3 minutes
current_pos = playback.make_coordinate(45.7)
print(f"Current position: {current_pos}")
```

</details>

In [ ]:
# Your solution here
